# 05_finetune: Fine-Tune a Small Model on Your Tickets

[Open in Colab](https://colab.research.google.com/github/Utkarsh-09/AI_GURU_labs/blob/main/notebooks/05_finetune.ipynb)

**Session:** Day 2, S11 — Lab: fine-tune it (90-minute block)
**Expected runtime:** about 40 minutes end to end on a Colab free-tier **T4**. **Training is budgeted at 25 minutes** and the notebook enforces it: at the budget it stops cleanly and saves what it has. The rest is install (1 min), model download (1–2 min), and the before/after checks (5 min).
**Needs:** a GPU runtime — *Runtime → Change runtime type → T4 GPU*, free tier, never Pro. No API key, no Hugging Face account. Reads your `train_clean.jsonl` / `val_clean.jsonl` from notebook 04 (on Drive); if they are not there it rebuilds them from `data/finetune/`. Downloads about 2.5 GB of model weights.
**A correct result looks like:** the training loss printed step by step falls from about **TBD** to below **TBD**; the saved adapter loads back onto a fresh base model and answers the five preview tickets with **5 of 5 schema-valid JSON records** (the untuned model manages fewer, and wordier); and the final cell prints `ADAPTER READY` with the adapter's two locations — your checkpoint folder (Google Drive on Colab) and `checkpoints/` in the repo.

> All data in this lab is synthetic. No real OQ material anywhere.

---
**The plan.** Load the cleaned dataset → look at exactly what the model will read → load a small open model in 4-bit → ask it five tickets *before* training → choose the adapter (**TODO 1**) → choose the training settings (**TODO 2**) → train, with checkpoints on Drive → read the loss → save → load the adapter back from disk and ask the same five tickets *after*.

**What you are deciding in this lab:** rank, alpha, which layers, learning rate, epochs, batch size. **What you are not doing:** plumbing. Tokenising, padding, checkpoint handling and the time budget live in `notebooks/finetune_utils.py` — open it if you are curious, it is short.

**If the runtime disconnects:** reconnect, then *Runtime → Run all*. Finished work is loaded from Drive, and training continues from the last checkpoint — it does not start again.

**No GPU today?** Set `SMOKE_TEST = True` in the settings cell. A tiny model on 20 rows runs on a CPU in about three minutes. It proves every step works; it does not learn the task.

**Why this cell:** one notebook has to run in two places — Colab during
the program, and a local machine as the fallback. This first code cell
detects which one it woke up in, mounts Google Drive on Colab so
checkpoints survive a disconnect, and sets the three variables every
later cell can rely on: `IN_COLAB`, `REPO_ROOT`, `CHECKPOINT_DIR`.

In [ ]:
# Environment detection: Colab vs local. Sets IN_COLAB, REPO_ROOT, CHECKPOINT_DIR.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The repo URL participants clone in Colab. Set once, here.
REPO_URL = "https://github.com/Utkarsh-09/AI_GURU_labs.git"

if IN_COLAB:
    # Drive first: checkpoints survive a runtime disconnect.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_ROOT = Path("/content/oq-advanced-ai")
    if not REPO_ROOT.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/oq-advanced-ai-checkpoints")
else:
    # Local: find the repo root by walking up until BUILD_SPEC.md appears.
    here = Path.cwd()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "BUILD_SPEC.md").exists()),
        here,
    )
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "local"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Make repo modules importable: config.endpoints, notebooks/utils.py
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

**Why this cell:** Colab already ships `torch`, `transformers`, `peft` and `accelerate`, at exactly the versions pinned in `requirements-finetune.txt` — so they are *not* reinstalled. Reinstalling them is how a cold runtime loses ten minutes and ends up asking for a restart. The one thing missing is `bitsandbytes`, the library that holds the base model in 4-bit. One prebuilt 43 MB wheel, exact pin.

In [ ]:
# Pinned install — the version matches requirements-finetune.txt. Colab only;
# a local machine installs requirements-finetune.txt once, during setup.
if IN_COLAB:
    %pip install -q bitsandbytes==0.50.2
print("Install cell done.")

**Why this cell:** every choice that changes *what gets trained* sits in one place, at the top, so a run can be described in one screen. `MODEL_NAME` picks the base model (`finetune_utils.MODEL_CHOICES` lists them). `RUN_NAME` names the folder on Drive where this run's checkpoints live — change it when you want a second experiment *next to* the first instead of on top of it.

In [ ]:
import torch

import finetune_utils

MODEL_NAME = "llama3.2-1b"    # see finetune_utils.MODEL_CHOICES
RUN_NAME = "run1"                # a new name = a new, separate experiment
SMOKE_TEST = False               # True = tiny model, 20 rows, CPU-friendly plumbing test

TRAINING_BUDGET_MINUTES = 25     # the lab's promise. Training stops itself here.
MAX_LENGTH = 1024                # longest row allowed, in tokens
SEED = 42

# Facilitator switch: lets an automated test flip the smoke test on.
if os.environ.get("OQ_SMOKE_TEST") == "1":
    SMOKE_TEST = True
if SMOKE_TEST:
    MODEL_NAME = "smoke-test"

USE_GPU = torch.cuda.is_available()
model_choice = finetune_utils.MODEL_CHOICES[MODEL_NAME]
run_dir = CHECKPOINT_DIR / "05_finetune" / f"{MODEL_NAME}_{RUN_NAME}"

print(f"model      : {MODEL_NAME}  ({model_choice['hf_repo']})")
print(f"run folder : {run_dir}")
if USE_GPU:
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
elif SMOKE_TEST:
    print("GPU        : none - fine, this is a smoke test")
else:
    TRAINING_BUDGET_MINUTES = 24 * 60
    print("GPU        : NONE. A full run on a CPU takes hours, so the 25-minute budget is switched off.")
    print("             In Colab: Runtime > Change runtime type > T4 GPU, then Run all.")

**Why this cell:** the model learns from the dataset *you* cleaned in notebook 04 — 373 training rows and 72 validation rows once the planted duplicates, leaks and broken records are out. If your group did not finish notebook 04, the helper rebuilds the same cleaned set from the committed files, so nobody is locked out of this lab. The 20 held-out tickets are loaded too, but only to *ask* the model questions. They are never trained on.

In [ ]:
import dataset_utils

train_pairs, val_pairs, data_source = finetune_utils.load_clean_pairs(REPO_ROOT, CHECKPOINT_DIR)
heldout_pairs = dataset_utils.load_jsonl(REPO_ROOT / "data" / "eval" / "heldout_20.jsonl")
schema = dataset_utils.load_schema(REPO_ROOT / "data" / "finetune" / "ticket_schema.json")

if SMOKE_TEST:
    train_pairs = train_pairs[:finetune_utils.SMOKE_TEST_TRAIN_ROWS]
    val_pairs = val_pairs[:finetune_utils.SMOKE_TEST_VAL_ROWS]

print(f"source   : {data_source}")
print(f"train    : {len(train_pairs)} rows")
print(f"val      : {len(val_pairs)} rows")
print(f"held-out : {len(heldout_pairs)} rows (for asking, never for training)")

**Why this cell:** look at one training row before spending GPU time on 373 of them. Three messages: the fixed system prompt, one messy ticket, and the record we want back — one line of JSON and nothing else. Whatever is in that third message is what the model will learn to produce, mistakes included. That is why notebook 04 came first.

In [ ]:
example_pair = train_pairs[0]
for message in example_pair["messages"]:
    shown_text = message["content"]
    if message["role"] == "system":
        shown_text = shown_text[:160] + " ...[system prompt continues]"
    print(f"--- {message['role']} ---")
    print(shown_text)
    print()

**Why this cell:** a chat model never sees "messages". It sees one long string with special marker tokens between the roles, produced by a *chat template*. The template used in training has to be the same one used when the model is served, token for token — and here it would not be. Your tuned model will be served by Ollama, and Hugging Face's template for this model adds a `Today Date:` line that Ollama's does not. Train with it and the model learns a header it will never see again. So the helper swaps in the text Ollama actually sends. Below is one full row exactly as the model reads it.

In [ ]:
tokenizer = finetune_utils.load_tokenizer(model_choice["hf_repo"])
template_note = finetune_utils.use_serving_template(tokenizer)
print(template_note)
print()

rendered_text = tokenizer.apply_chat_template(example_pair["messages"], tokenize=False)
print(rendered_text[:330])
print("   ...")
print(rendered_text[-300:])

**Why this cell:** training cost is counted in tokens, not rows. Each row becomes token ids plus a *loss mask*: the model reads the whole row but is graded only on the answer. Without the mask most of the training signal would go into memorising the system prompt, which is the same 250 tokens in every row. The two totals printed here are the ones that decide how long training takes (all tokens) and how much there is to learn from (graded tokens).

In [ ]:
train_dataset = finetune_utils.build_token_dataset(train_pairs, tokenizer, MAX_LENGTH)
val_dataset = finetune_utils.build_token_dataset(val_pairs, tokenizer, MAX_LENGTH)

train_tokens, train_graded_tokens = finetune_utils.count_tokens(train_dataset)
longest_row = max(len(row["input_ids"]) for row in train_dataset.rows)

print(f"train rows            : {len(train_dataset)}")
print(f"tokens per epoch      : {train_tokens:,}")
print(f"  of which are graded : {train_graded_tokens:,}  ({train_graded_tokens / train_tokens:.0%} - the JSON answers)")
print(f"longest row           : {longest_row} tokens (limit {MAX_LENGTH})")

**Why this cell:** this is the **Q** in QLoRA. On a GPU the base model is loaded with its weights squeezed to 4 bits each, which is what lets it sit on a free T4 with room left over for training. Those weights are frozen — nothing in this lab changes them. (On a CPU the helper loads ordinary 32-bit weights instead, because 4-bit needs a GPU. The adapter that comes out has the same format either way.)

In [ ]:
base_model = finetune_utils.load_base_model(model_choice["hf_repo"], USE_GPU)

_, base_parameter_count = finetune_utils.count_parameters(base_model)
memory_gb = base_model.get_memory_footprint() / 1e9
print(f"loaded     : {model_choice['hf_repo']}")
print(f"parameters : {base_parameter_count / 1e6:,.0f} M  (4-bit layers count packed, so this reads low on a GPU)")
print(f"in memory  : {memory_gb:.2f} GB on {'GPU, 4-bit' if USE_GPU else 'CPU, 32-bit'}")

**Why this cell (milestone 1):** measure before you change anything. Five held-out tickets go to the *untuned* model, and each reply is checked the strict way: does `json.loads` accept it as it stands, and does it pass the ticket schema? Read the replies, not just the score — look for long-winded `requested_action`s and invented queue names. These replies are saved, so after a reconnect this cell loads them instead of asking again.

In [ ]:
import utils

PREVIEW_COUNT = 5
preview_pairs = heldout_pairs[:PREVIEW_COUNT]

before_replies = utils.load_json(run_dir, "replies_before", default=None)
if before_replies is None:
    before_replies = []
    for pair in preview_pairs:
        reply_text = finetune_utils.generate_reply(base_model, tokenizer, pair["messages"][:2])
        before_replies.append({"ticket_id": pair["ticket_id"], "reply": reply_text})
    utils.save_json(run_dir, "replies_before", before_replies)

before_valid_count = 0
for item in before_replies:
    verdict = finetune_utils.check_reply(item["reply"], schema)
    before_valid_count += verdict["valid"]
    print(f"{item['ticket_id']}  schema-valid: {verdict['valid']}")
    print(f"  {item['reply'][:300]}")
print()
print(f"BEFORE training: {before_valid_count} of {len(before_replies)} replies are schema-valid")

**Why this cell — TODO 1, the adapter:** LoRA leaves the base model frozen and trains two small matrices beside each layer you name. Four decisions:

- `LORA_RANK` — the width of those matrices. Higher = more capacity, bigger file, slower. 8 to 32 is the normal range; this is a *format* task, which needs little.
- `LORA_ALPHA` — how loudly the adapter speaks relative to the base model. The adapter's output is scaled by `alpha / rank`; a common starting point is alpha = rank or 2 × rank.
- `LORA_DROPOUT` — regularisation. With only 373 rows a little (0.05) is reasonable.
- `TARGET_MODULES` — which layers get an adapter. Attention only (`q_proj k_proj v_proj o_proj`) is lighter; adding the MLP layers (`gate_proj up_proj down_proj`) is what the QLoRA paper found matters most for quality.

The cell prints how many parameters you are actually training. Expect around 1% of the model.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── TODO 1 ───────────────────────────────────────────────────────── (solution)
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# ───────────────────────────────────────────────────────────────────

assert LORA_RANK is not ... and LORA_ALPHA is not ..., "TODO 1 is not filled in yet"
assert LORA_DROPOUT is not ... and TARGET_MODULES is not ..., "TODO 1 is not filled in yet"

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

if USE_GPU:
    # Housekeeping a 4-bit model needs before it can be trained.
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=False)

model = get_peft_model(base_model, lora_config)

trainable_count, total_count = finetune_utils.count_parameters(model)
print(f"rank {LORA_RANK}, alpha {LORA_ALPHA} (scale {LORA_ALPHA / LORA_RANK:.1f}), dropout {LORA_DROPOUT}")
print(f"adapters on : {', '.join(TARGET_MODULES)}")
print(f"training    : {trainable_count:,} of {total_count:,} parameters ({trainable_count / total_count:.2%})")

**Why this cell — TODO 2, the training settings:** four numbers decide how long training takes and how well it goes.

- `NUM_EPOCHS` — passes over the 373 rows. Too few and the format is shaky; too many and the model memorises the training tickets. Small datasets usually want 2 to 4.
- `LEARNING_RATE` — step size. LoRA wants far more than full fine-tuning does: `2e-4` is the usual starting point. `2e-5` will look like nothing is happening; `2e-3` will likely blow up.
- `BATCH_SIZE` — rows per forward pass. Limited by GPU memory. 4 is safe on a T4.
- `GRADIENT_ACCUMULATION` — how many batches to add up before one update. `BATCH_SIZE × GRADIENT_ACCUMULATION` is the *effective* batch; 16 is a sensible target.

The cell prints the number of update steps. **Time scales with `NUM_EPOCHS`** — that is the one that spends your 25 minutes.

In [ ]:
# ── TODO 2 ───────────────────────────────────────────────────────── (solution)
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
# ───────────────────────────────────────────────────────────────────

assert NUM_EPOCHS is not ... and LEARNING_RATE is not ..., "TODO 2 is not filled in yet"
assert BATCH_SIZE is not ... and GRADIENT_ACCUMULATION is not ..., "TODO 2 is not filled in yet"

import math

effective_batch = BATCH_SIZE * GRADIENT_ACCUMULATION
steps_per_epoch = math.ceil(len(train_dataset) / effective_batch)
total_steps = steps_per_epoch * NUM_EPOCHS
print(f"effective batch : {effective_batch} rows per update")
print(f"updates         : {steps_per_epoch} per epoch x {NUM_EPOCHS} epochs = {total_steps} steps")
print(f"tokens to read  : {train_tokens * NUM_EPOCHS:,}")

**Why this cell:** checkpoints are only useful if they belong to *this* experiment. The helper writes your settings into the run folder the first time, and on every later visit compares them. Same settings → it says `resume` (or `finished`) and no work is repeated. Different settings → it stops and asks for a new `RUN_NAME`, rather than quietly continuing a rank-16 run as if it were your new rank-8 idea.

In [ ]:
run_settings = {
    "model": model_choice["hf_repo"],
    "smoke_test": SMOKE_TEST,
    "train_rows": len(train_dataset),
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
}
run_status = finetune_utils.check_run_folder(run_dir, run_settings)
last_checkpoint = finetune_utils.find_last_checkpoint(run_dir)

print(f"run folder : {run_dir}")
print(f"status     : {run_status}")
print(f"checkpoint : {last_checkpoint.name if last_checkpoint else 'none yet'}")

**Why this cell:** your four numbers go into `TrainingArguments`; everything else here is plumbing chosen once for a free T4 and a flaky connection. The parts worth knowing: `save_steps` writes a checkpoint to the run folder — on Drive — every few updates, and `save_total_limit` keeps only the newest two so Drive does not fill up. `fp16` because a T4 has no bfloat16. `gradient_checkpointing` trades a little speed for a lot of memory. `group_by_length` batches similar-length tickets together so less time is wasted on padding.

In [ ]:
from transformers import PrinterCallback, Trainer, TrainingArguments

SAVE_EVERY_STEPS = 2 if SMOKE_TEST else 10

training_arguments = TrainingArguments(
    output_dir=str(run_dir),
    # --- your choices from TODO 2 ---
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    # --- the schedule around them ---
    lr_scheduler_type="cosine",
    warmup_steps=0.1,                  # a fraction: the first 10% of steps ramp up
    weight_decay=0.0,
    max_grad_norm=1.0,
    # --- surviving a disconnect ---
    save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS,
    save_total_limit=2,
    # --- seeing what happens ---
    logging_strategy="steps",
    logging_steps=1 if SMOKE_TEST else 2,
    logging_first_step=True,
    eval_strategy="epoch",
    per_device_eval_batch_size=BATCH_SIZE,
    report_to="none",
    disable_tqdm=True,
    # --- fitting on a T4 ---
    fp16=USE_GPU,
    use_cpu=not USE_GPU,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    train_sampling_strategy="group_by_length",
    optim="adamw_torch",
    dataloader_pin_memory=USE_GPU,
    remove_unused_columns=False,
    seed=SEED,
)

progress = finetune_utils.ProgressCallback(run_dir, budget_minutes=TRAINING_BUDGET_MINUTES)
trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=finetune_utils.PadCollator(tokenizer.pad_token_id),
    processing_class=tokenizer,
    callbacks=[progress],
)
trainer.remove_callback(PrinterCallback)   # `progress` prints the loss instead, one tidy line per step
print(f"trainer ready: {total_steps} steps, a checkpoint every {SAVE_EVERY_STEPS}, budget {TRAINING_BUDGET_MINUTES} min")

**Why this cell (milestone 2) — the 25 minutes:** this is the only slow cell. Watch the `loss` column: it should drop fast in the first few steps (the model discovers the JSON format) and then grind down slowly (it learns *your* conventions — which queue, which urgency). The last column projects the total time from the pace so far. If the run hits the budget it stops itself and keeps what it learned.

**Disconnected?** Reconnect and *Run all*. This cell finds the newest checkpoint in the run folder and continues from that step. If the run had already finished, it does nothing at all.

In [ ]:
if run_status == "finished":
    print("This run already finished - its adapter is in the run folder. Nothing to train.")
else:
    trainer.train(resume_from_checkpoint=str(last_checkpoint) if last_checkpoint else None)
    print()
    print(f"training stopped at step {trainer.state.global_step} of {trainer.state.max_steps}"
          f"{'  (time budget reached)' if progress.stopped_by_budget else ''}")

**Why this cell (milestone 3):** the adapter is the deliverable, so it is saved twice. Once in the run folder — on Google Drive when you are in Colab, so it outlives the runtime. Once under `checkpoints/` in the repo, which is where notebook 06 and the `tuned` endpoint look for it. Note the size: tens of megabytes. The base model it rides on is thousands. That ratio is why adapters are practical to version, ship and swap.

In [ ]:
import shutil

final_adapter_dir = run_dir / finetune_utils.FINAL_ADAPTER_FOLDER
repo_adapter_dir = REPO_ROOT / "checkpoints" / f"adapter_{MODEL_NAME}_{RUN_NAME}"

if run_status != "finished":
    finetune_utils.save_adapter(model, tokenizer, final_adapter_dir)

shutil.copytree(final_adapter_dir, repo_adapter_dir, dirs_exist_ok=True)

adapter_megabytes = (final_adapter_dir / "adapter_model.safetensors").stat().st_size / 1e6
print(f"adapter saved : {final_adapter_dir}")
print(f"and copied to : {repo_adapter_dir}")
print(f"size          : {adapter_megabytes:.1f} MB")
print(f"files         : {sorted(path.name for path in final_adapter_dir.iterdir())}")

**Why this cell:** the loss log was written to the run folder as training went, so this table is complete even if the run was done in two halves across a disconnect. Two things to read. **Training loss**: did it fall and flatten, or is it still dropping (more epochs would help) or bouncing (learning rate too high)? **Validation loss**, once per epoch, on tickets the model never trained on: if it starts *rising* while training loss keeps falling, the model has begun memorising — that is your signal for how many epochs is enough.

In [ ]:
loss_log = finetune_utils.load_loss_log(run_dir)
train_entries = [entry for entry in loss_log if "loss" in entry]
eval_entries = [entry for entry in loss_log if "eval_loss" in entry]

print("step   epoch   train loss   minutes")
show_every = max(len(train_entries) // 12, 1)
for position, entry in enumerate(train_entries):
    is_last = position == len(train_entries) - 1
    if position % show_every == 0 or is_last:
        print(f"{entry['step']:>4}   {entry['epoch']:>5.2f}   {entry['loss']:>10.4f}   {entry['minutes']:>7.1f}")

print()
print("validation loss at the end of each epoch:")
for entry in eval_entries:
    print(f"  epoch {entry['epoch']:>4.1f}   eval loss {entry['eval_loss']:.4f}")

first_loss = train_entries[0]["loss"]
last_loss = train_entries[-1]["loss"]
training_minutes = train_entries[-1]["minutes"]
print()
print(f"training loss {first_loss:.3f} -> {last_loss:.3f} in {training_minutes:.1f} minutes of training")

**Why this cell (milestone 4) — the honest test:** the model in memory is not the deliverable; the files on disk are. So throw the trained model away, load a *fresh* base model, attach the adapter from disk, and ask the same five tickets as before. If this works, the adapter folder is genuinely all anyone needs. Compare with the BEFORE replies above: look at `requested_action` (short and imperative now?) and `routing_queue` (a real queue name now?).

In [ ]:
import gc

# Free the memory the training objects hold. Plain `del` would fail on a
# second run of this cell, so drop the names only if they still exist.
for name in ["trainer", "model", "base_model"]:
    globals().pop(name, None)
gc.collect()
if USE_GPU:
    torch.cuda.empty_cache()

tuned_model, tuned_tokenizer = finetune_utils.load_adapter(model_choice["hf_repo"], final_adapter_dir, USE_GPU)

after_replies = []
after_valid_count = 0
for pair in preview_pairs:
    reply_text = finetune_utils.generate_reply(tuned_model, tuned_tokenizer, pair["messages"][:2])
    verdict = finetune_utils.check_reply(reply_text, schema)
    after_valid_count += verdict["valid"]
    after_replies.append({"ticket_id": pair["ticket_id"], "reply": reply_text, "valid": verdict["valid"]})
    print(f"{pair['ticket_id']}  schema-valid: {verdict['valid']}")
    print(f"  tuned    : {reply_text[:300]}")
    print(f"  expected : {dataset_utils.pair_completion_text(pair)}")

utils.save_json(run_dir, "replies_after", after_replies)
print()
print(f"BEFORE training: {before_valid_count} of {len(before_replies)} schema-valid")
print(f"AFTER  training: {after_valid_count} of {len(after_replies)} schema-valid   (adapter loaded back from disk)")

**Why this cell (optional, never blocks you):** everything after today reaches your tuned model through one name — the `tuned` endpoint in `config/endpoints.py`, which is an Ollama model called `oq-ticket-tuned`. Registering is two lines of Modelfile (`FROM` the same base model, `ADAPTER` your folder) and one `ollama create`; `scripts/register_adapter.py` does exactly that and prints the Modelfile so you can see it. If Ollama is not running where this notebook runs — normal on a Colab GPU runtime — the cell says so and moves on. Notebook 06 starts Ollama and runs the same script.

In [ ]:
import subprocess

register_script = REPO_ROOT / "scripts" / "register_adapter.py"

if SMOKE_TEST:
    print("Smoke test: the tiny model has no Ollama twin, so there is nothing to register.")
elif shutil.which("ollama") is None:
    print("Ollama is not installed in this runtime - skipped. Notebook 06 registers the adapter.")
else:
    command = [sys.executable, str(register_script), "--adapter", str(repo_adapter_dir)]
    completed = subprocess.run(command, capture_output=True, text=True)
    print(completed.stdout + completed.stderr)
    if completed.returncode != 0:
        print("Not registered (see the message above). This does not affect your adapter - notebook 06 will try again.")

**Why this cell:** the declared result, in one block, so "done" can be checked at a glance from across the room. Five tickets is a taste, not a measurement — the real score comes next, in notebook 06, where the eval harness runs all 20 held-out tickets against base and tuned side by side.

In [ ]:
summary = {
    "model": model_choice["hf_repo"],
    "ollama_base": model_choice["ollama_base"],
    "run_folder": str(run_dir),
    "adapter_on_drive": str(final_adapter_dir),
    "adapter_in_repo": str(repo_adapter_dir),
    "adapter_megabytes": round(adapter_megabytes, 1),
    "train_rows": len(train_dataset),
    "steps_done": train_entries[-1]["step"],
    "steps_planned": total_steps,
    "first_loss": first_loss,
    "last_loss": last_loss,
    "training_minutes": training_minutes,
    "valid_before": before_valid_count,
    "valid_after": after_valid_count,
    "preview_tickets": len(preview_pairs),
}
utils.save_json(run_dir, "05_summary", summary)

print()
print("ADAPTER READY" if not SMOKE_TEST else "SMOKE TEST COMPLETE (plumbing only - this tiny model has not learned the task)")
print(f"  base model        : {summary['model']}")
print(f"  trained           : {summary['steps_done']} of {summary['steps_planned']} steps, "
      f"{summary['training_minutes']:.1f} minutes, on {'GPU' if USE_GPU else 'CPU'}")
print(f"  training loss     : {summary['first_loss']:.3f} -> {summary['last_loss']:.3f}")
print(f"  schema-valid JSON : {summary['valid_before']} of {summary['preview_tickets']} before -> "
      f"{summary['valid_after']} of {summary['preview_tickets']} after")
print(f"  adapter           : {summary['adapter_megabytes']} MB")
print(f"    on Drive / run folder : {summary['adapter_on_drive']}")
print(f"    in the repo           : {summary['adapter_in_repo']}")
print("  next              : notebook 06 scores it on all 20 held-out tickets against the base model")